## descSPIM-SLIM: Python Control

This code offers a way to control descSPIM-SLIM via a Python-based graphical user interface (GUI), facilitating straightforward 3D acquisition/imaging. Please install it in your environment by following the "descSPIM-SLIM: Python Control Installation Instructions".

SAMPLE_SERIAL = "26006224" # sample stage
CAMERA_SERIAL = "26006304" # camera stage

These are the serial numbers of the KST201 controllers.
They differ for each individual device. Please replace them with the serial numbers printed on your own controller units.

### Homing

In [ ]:

import time
import threading
from pathlib import Path
import tkinter as tk
from tkinter import ttk, messagebox

# ----- Python .NET bridge -----
import clr

# ===== User-configurable constants =====
KINESIS_DIR = Path(r"C:\Program Files\Thorlabs\Kinesis")
SAMPLE_SERIAL = "26006224"  # sample stage
CAMERA_SERIAL = "26006304"  # camera stage
POLL_INTERVAL_MS = 250
HOME_TIMEOUT_MS = 60_000

# ===== Load Kinesis assemblies =====
clr.AddReference(str(KINESIS_DIR / "Thorlabs.MotionControl.DeviceManagerCLI.dll"))
clr.AddReference(str(KINESIS_DIR / "Thorlabs.MotionControl.GenericMotorCLI.dll"))
clr.AddReference(str(KINESIS_DIR / "Thorlabs.MotionControl.KCube.StepperMotorCLI.dll"))

from Thorlabs.MotionControl.DeviceManagerCLI import DeviceManagerCLI
from Thorlabs.MotionControl.KCube.StepperMotorCLI import KCubeStepper


def connect_stepper(serial: str) -> KCubeStepper:
    """Create, connect, start polling, enable, and best-effort load motor configuration."""
    DeviceManagerCLI.BuildDeviceList()
    dev = KCubeStepper.CreateKCubeStepper(serial)
    dev.Connect(serial)
    time.sleep(0.2)
    dev.StartPolling(POLL_INTERVAL_MS)
    time.sleep(0.2)
    dev.EnableDevice()
    time.sleep(0.5)
    try:
        dev.LoadMotorConfiguration(serial)
    except Exception:
        # Some versions require explicit sources; proceed with defaults if not available
        pass
    return dev


def home_device(dev: KCubeStepper, timeout_ms: int):
    """Home the device; assumes synchronous Home(timeout_ms) is available."""
    dev.Home(timeout_ms)


def disconnect_stepper(dev: KCubeStepper):
    """Stop polling and disconnect; ignore minor errors."""
    if not dev:
        return
    try:
        dev.StopPolling()
    except Exception:
        pass
    try:
        dev.Disconnect()
    except Exception:
        pass


class MinimalHomingApp(tk.Tk):
    """Minimal GUI with exactly one actionable button: 'Home'."""

    def __init__(self):
        super().__init__()
        self.title("Kinesis KCube Stepper - Homing (Minimal)")
        self.geometry("460x200")
        self.resizable(False, False)

        # Device handles (None until connected)
        self.sample_dev = None
        self.camera_dev = None

        # UI
        self.status_var = tk.StringVar(value="Ready. Click 'Home' to home both devices.")
        self._build_ui()

        # Close handler
        self.protocol("WM_DELETE_WINDOW", self.on_close)

    def _build_ui(self):
        wrapper = ttk.Frame(self, padding=16)
        wrapper.pack(fill="both", expand=True)

        # Big single button
        self.home_btn = ttk.Button(wrapper, text="Home", command=self.on_home_clicked)
        self.home_btn.pack(pady=12, ipadx=24, ipady=8)

        # Status label (informational only; not a control)
        ttk.Label(wrapper, textvariable=self.status_var, wraplength=420).pack(pady=(8, 0))

        # Serial label (informational only)
        ttk.Label(
            wrapper,
            text=f"Sample: {SAMPLE_SERIAL}   |   Camera: {CAMERA_SERIAL}"
        ).pack(pady=(6, 0))

    def set_busy(self, busy: bool):
        self.home_btn.configure(state="disabled" if busy else "normal")

    def on_home_clicked(self):
        """Single action: ensure connected, then home both devices."""
        def worker():
            self.set_busy(True)
            try:
                self.status_var.set("Connecting devices (if not connected)...")
                # Connect on demand
                if self.sample_dev is None:
                    self.sample_dev = connect_stepper(SAMPLE_SERIAL)
                if self.camera_dev is None:
                    self.camera_dev = connect_stepper(CAMERA_SERIAL)

                self.status_var.set("Homing sample stage...")
                home_device(self.sample_dev, HOME_TIMEOUT_MS)

                self.status_var.set("Homing camera stage...")
                home_device(self.camera_dev, HOME_TIMEOUT_MS)

                self.status_var.set("Homing complete.")
            except Exception as e:
                self.status_var.set(f"Error: {e}")
                messagebox.showerror("Homing Error", str(e))
            finally:
                self.set_busy(False)

        threading.Thread(target=worker, daemon=True).start()

    def on_close(self):
        """Gracefully disconnect on exit."""
        try:
            disconnect_stepper(self.sample_dev)
            disconnect_stepper(self.camera_dev)
        finally:
            self.destroy()


if __name__ == "__main__":
    app = MinimalHomingApp()
    app.mainloop()


### Set verocity and acceleration for focusing 
It is recommended to set the parameters to [sample] MaxVelocity = 0.1 mm/s, Acceleration = 2 and [camera] MaxVelocity = 0.0342 mm/s, Acceleration = 2 for optimal focus alignment.
The relative velocity of the camera stage is determined as a constant multiple of the sample stage speed, derived from the difference in refractive indices between air and the CUBIC gel.
For detailed methodology, please refer to Otomo K et al., Nature Communications, Methods section.

In [ ]:
import time
import threading
from pathlib import Path

# --- GUI ---
import tkinter as tk
from tkinter import ttk

# --- pythonnet / Kinesis ---
import clr
from System import Decimal as SysDecimal
from System.Globalization import CultureInfo

def mkdec(x) -> SysDecimal:
    """Use Decimal.Parse with invariant culture to avoid pythonnet implicit conversion pitfalls."""
    return SysDecimal.Parse(str(x), CultureInfo.InvariantCulture)

# ===== Kinesis assemblies =====
KINESIS_DIR = Path(r"C:\Program Files\Thorlabs\Kinesis")
clr.AddReference(str(KINESIS_DIR / "Thorlabs.MotionControl.DeviceManagerCLI.dll"))
clr.AddReference(str(KINESIS_DIR / "Thorlabs.MotionControl.GenericMotorCLI.dll"))
clr.AddReference(str(KINESIS_DIR / "Thorlabs.MotionControl.KCube.StepperMotorCLI.dll"))

from Thorlabs.MotionControl.DeviceManagerCLI import DeviceManagerCLI
from Thorlabs.MotionControl.KCube.StepperMotorCLI import KCubeStepper
from Thorlabs.MotionControl.GenericMotorCLI.ControlParameters import VelocityParameters

# ===== Device configuration =====
SER_SAMPLE = "26006224"    # sampleStage
SER_CAMERA = "26006304"    # cameraStage
DU_PER_MM = mkdec(2008624) # 1 mm = 2,008,624 DU

def reconnect_kcube(serial: str, retries: int = 3, wait_s: float = 1.0) -> KCubeStepper:
    """
    Robustly (re)connect a KCube stepper.
    Sequence:
      BuildDeviceList -> Create -> Connect -> WaitForSettingsInitialized
      -> StartPolling -> EnableDevice -> LoadMotorConfiguration
    Retries on failure.
    """
    last_err = None
    dev = None
    for _ in range(retries):
        try:
            DeviceManagerCLI.BuildDeviceList()
            dev = KCubeStepper.CreateKCubeStepper(serial)
            time.sleep(0.2)
            dev.Connect(serial)
            try:
                dev.WaitForSettingsInitialized(5000)
            except Exception:
                pass
            time.sleep(0.2)
            dev.StartPolling(250)
            time.sleep(0.2)
            dev.EnableDevice()
            time.sleep(0.3)
            try:
                dev.LoadMotorConfiguration(serial)
            except Exception:
                pass
            print(f"[OK] Connected {serial}")
            return dev
        except Exception as e:
            last_err = e
            # best-effort cleanup before retry
            try:
                if dev is not None:
                    dev.StopPolling()
            except Exception:
                pass
            try:
                if dev is not None:
                    dev.Disconnect()
            except Exception:
                pass
            time.sleep(wait_s)
    raise RuntimeError(f"[{serial}] Reconnection failed: {last_err!r}")

def set_speed_mm(dev, vel_mm_s: float, acc_mm_s2: float):
    """
    Set velocity/acceleration in mm/s and mm/s^2.
    Prefer real-units API; if not available, fall back to device units (DU).
    """
    vp = VelocityParameters()
    # Try real units
    try:
        vp.MaxVelocity  = mkdec(vel_mm_s)
        vp.Acceleration = mkdec(acc_mm_s2)
        dev.SetVelocityParams(vp)
        return
    except Exception:
        pass
    # Fallback to DU
    try:
        vp.MaxVelocity  = SysDecimal.Multiply(mkdec(vel_mm_s),  DU_PER_MM)
        vp.Acceleration = SysDecimal.Multiply(mkdec(acc_mm_s2), DU_PER_MM)
        dev.SetVelocityParams(vp)
    except Exception as e:
        raise RuntimeError(f"Failed to set velocity parameters: {e!r}")

def get_speed(dev):
    """Return (MaxVelocity, Acceleration) from current device parameters (raw value from API)."""
    vp = dev.GetVelocityParams()
    return vp.MaxVelocity, vp.Acceleration

def disconnect_safe(dev, tag: str):
    """Stop polling and disconnect, ignoring minor errors."""
    if dev is None:
        return
    try:
        dev.StopPolling()
    except Exception:
        pass
    try:
        dev.Disconnect()
        print(f"[CLOSE] {tag} disconnected.")
    except Exception:
        pass

# ===== Minimal GUI with a single actionable button =====
class SpeedGUI(tk.Tk):
    def __init__(self):
        super().__init__()
        self.title("KCube Stepper - Velocity Setup (Minimal)")
        self.geometry("520x260")
        self.resizable(False, False)

        # Device handles (used only within a click; kept here for safety on window close)
        self.sample_dev = None
        self.camera_dev = None

        # Default values (same as example in your script)
        self.sample_vel = tk.StringVar(value="0.50")
        self.sample_acc = tk.StringVar(value="1.0")
        self.camera_vel = tk.StringVar(value="0.40")
        self.camera_acc = tk.StringVar(value="0.8")

        self.status = tk.StringVar(value="Ready. Enter speeds and click 'Set Speeds'.")

        self._build_ui()
        self.protocol("WM_DELETE_WINDOW", self.on_close)

    def _build_ui(self):
        pad = 10
        frm = ttk.Frame(self, padding=pad)
        frm.pack(fill="both", expand=True)

        # Serial info (labels only, not editable)
        ttk.Label(frm, text=f"Sample: {SER_SAMPLE}").grid(row=0, column=0, sticky="w", padx=pad, pady=(pad, 4))
        ttk.Label(frm, text=f"Camera: {SER_CAMERA}").grid(row=0, column=1, sticky="w", padx=pad, pady=(pad, 4))

        # Input grid
        ttk.Label(frm, text="Velocity (mm/s)").grid(row=1, column=0, sticky="w", padx=pad)
        ttk.Label(frm, text="Acceleration (mm/s²)").grid(row=1, column=1, sticky="w", padx=pad)

        # Sample inputs
        srow = 2
        ttk.Label(frm, text="Sample").grid(row=srow, column=0, sticky="e", padx=(pad, 4))
        sv_entry = ttk.Entry(frm, width=12, textvariable=self.sample_vel)
        sa_entry = ttk.Entry(frm, width=12, textvariable=self.sample_acc)
        sv_entry.grid(row=srow, column=0, sticky="w", padx=(140, 4))
        sa_entry.grid(row=srow, column=1, sticky="w", padx=(pad, 4))

        # Camera inputs
        crow = 3
        ttk.Label(frm, text="Camera").grid(row=crow, column=0, sticky="e", padx=(pad, 4), pady=(4, 0))
        cv_entry = ttk.Entry(frm, width=12, textvariable=self.camera_vel)
        ca_entry = ttk.Entry(frm, width=12, textvariable=self.camera_acc)
        cv_entry.grid(row=crow, column=0, sticky="w", padx=(140, 4), pady=(4, 0))
        ca_entry.grid(row=crow, column=1, sticky="w", padx=(pad, 4), pady=(4, 0))

        # Single action button
        self.apply_btn = ttk.Button(frm, text="Set Speeds", command=self.on_set_speeds_clicked)
        self.apply_btn.grid(row=4, column=0, columnspan=2, pady=(12, 4))

        # Status label
        ttk.Label(frm, textvariable=self.status, wraplength=480).grid(row=5, column=0, columnspan=2, padx=pad, pady=(4, pad), sticky="w")

    def set_busy(self, busy: bool):
        self.apply_btn.configure(state="disabled" if busy else "normal")

    def on_set_speeds_clicked(self):
        """Single-button workflow: connect -> set speeds -> read back -> disconnect."""
        def worker():
            self.set_busy(True)
            try:
                # Parse inputs
                s_vel = float(self.sample_vel.get().strip())
                s_acc = float(self.sample_acc.get().strip())
                c_vel = float(self.camera_vel.get().strip())
                c_acc = float(self.camera_acc.get().strip())

                self.status.set("Connecting devices...")
                self.sample_dev = reconnect_kcube(SER_SAMPLE)
                self.camera_dev = reconnect_kcube(SER_CAMERA)

                self.status.set("Setting sample velocity parameters...")
                set_speed_mm(self.sample_dev, vel_mm_s=s_vel, acc_mm_s2=s_acc)

                self.status.set("Setting camera velocity parameters...")
                set_speed_mm(self.camera_dev, vel_mm_s=c_vel, acc_mm_s2=c_acc)

                # Read back (raw values from the API; units depend on firmware/capabilities)
                sv, sa = get_speed(self.sample_dev)
                cv, ca = get_speed(self.camera_dev)

                print(f"[sample] MaxVelocity={sv}, Acceleration={sa}")
                print(f"[camera] MaxVelocity={cv}, Acceleration={ca}")

                self.status.set("Done. Parameters applied and devices will be disconnected.")
            except Exception as e:
                self.status.set(f"Error: {e}")
            finally:
                # Always disconnect at the end of the operation
                disconnect_safe(self.sample_dev, "sample")
                disconnect_safe(self.camera_dev, "camera")
                self.sample_dev = None
                self.camera_dev = None
                self.set_busy(False)

        threading.Thread(target=worker, daemon=True).start()

    def on_close(self):
        """Best-effort disconnect if window is closed mid-operation."""
        try:
            disconnect_safe(self.sample_dev, "sample")
            disconnect_safe(self.camera_dev, "camera")
        finally:
            self.destroy()

if __name__ == "__main__":
    app = SpeedGUI()
    app.mainloop()


### Synchronized Imaging Procedure
The following steps outline the configuration and initiation process for the synchronized imaging experiment.

#### ThorCam Configuration
First, open the Settings window in the ThorCam application. Set the Exposure Time to 300 ms and verify that the Frames per Trigger option is set to Continuous.

Next, configure the Hardware Triggering settings: set the Mode to None and the Polarity to On High. Close the Settings window once the configuration is complete.

#### System Preparation and Script Execution
After achieving focus, you must run the Jupyter notebook script first.

The script will prompt you to ARM (put in standby) ThorCam using the configuration: Standard / On High.

You must wait until the console displays the following confirmation and plan messages, indicating successful device connection and movement parameters:

[INFO] Please arm ThorCam in “Standard / On High” mode and keep it ready.
[OK] Connected 26006224
[OK] Connected 26006304
[PLAN] sample=aaa mm @ 0.02000 mm/s (T≈650.0s)
[PLAN] camera≈bbb mm @ 0.00684 mm/s


#### Synchronization and Recording
The final step is to wait until the console displays the starting position readout, for example:
[POS ] start: sample=xxx mm, camera=yyy mm

Once this position readout appears, immediately start the ThorCam recording to achieve synchronized imaging.

By executing the next cell, you can return both stages to the imaging start position.
This allows you to change the excitation wavelength and perform repeated imaging, enabling multicolor 3D imaging.
However, it is recommended to record the starting positions of both stages in advance as a precaution.

In [ ]:
# --- dual_move_minimal.py: Minimal code to isolate and verify motion reliability ---

import time, threading
from pathlib import Path
import clr

# ---- User environment ----
KINESIS_DIR = Path(r"C:\Program Files\Thorlabs\Kinesis")
SER_SAMPLE  = "26006224"   # sampleStage
SER_CAMERA  = "26006304"   # cameraStage
DU_PER_MM   = 2008624      # conversion factor for your environment

# Target: sample 13 mm / 0.02 mm/s, camera 0.00684 mm/s (start simultaneously)
sample_dist_mm   = 13.0
sample_speed_mm  = 0.020
camera_speed_mm  = 0.00684
move_time_s      = sample_dist_mm / sample_speed_mm
camera_dist_mm   = camera_speed_mm * move_time_s

# ---- DLL loading (legacy CLI) ----
clr.AddReference(str(KINESIS_DIR / "Thorlabs.MotionControl.DeviceManagerCLI.dll"))
clr.AddReference(str(KINESIS_DIR / "Thorlabs.MotionControl.GenericMotorCLI.dll"))
clr.AddReference(str(KINESIS_DIR / "Thorlabs.MotionControl.KCube.StepperMotorCLI.dll"))
from Thorlabs.MotionControl.DeviceManagerCLI import DeviceManagerCLI
from Thorlabs.MotionControl.KCube.StepperMotorCLI import KCubeStepper
from System import Decimal as SysDecimal
from System.Globalization import CultureInfo

def dec(x):  # create System.Decimal
    return SysDecimal.Parse(str(x), CultureInfo.InvariantCulture)

def connect_kst_robust(serial: str, tries=8, pause_s=0.5):
    last = None
    for i in range(tries):
        try:
            DeviceManagerCLI.BuildDeviceList()
            dev = KCubeStepper.CreateKCubeStepper(serial)
            dev.Connect(serial)
            try: dev.WaitForSettingsInitialized(5000)
            except Exception: pass
            time.sleep(0.2); dev.StartPolling(200)
            time.sleep(0.2); dev.EnableDevice()
            time.sleep(0.3)
            try: dev.LoadMotorConfiguration(serial)
            except Exception: pass
            print(f"[OK] Connected {serial}")
            return dev
        except Exception as e:
            last = e
            print(f"[Retry {i+1}/{tries}] connect {serial} failed: {e}")
            time.sleep(pause_s)
    raise last

def get_pos_mm(dev) -> float:
    return float(str(dev.Position))

def wait_complete(dev, timeout_s=3600):
    try:
        dev.WaitForMoveToComplete(int(timeout_s*1000)); return
    except Exception:
        pass
    t0 = time.time()
    while time.time() - t0 < timeout_s:
        try:
            if hasattr(dev, "IsDeviceBusy") and not bool(dev.IsDeviceBusy):
                break
        except Exception:
            pass
        time.sleep(0.02)

def set_speed_mm_s(dev, vel_mm_s: float, acc_mm_s2: float):
    # absorb namespace differences of VelocityParameters
    for ns in (
        "Thorlabs.MotionControl.GenericMotorCLI.ControlParameters",
        "Thorlabs.MotionControl.GenericMotorCLI.Settings",
        "Thorlabs.MotionControl.GenericMotorCLI",
    ):
        try:
            mod = __import__(ns, fromlist=["VelocityParameters"])
            VelocityParameters = getattr(mod, "VelocityParameters")
            vp = VelocityParameters()
            vp.MaxVelocity  = dec(vel_mm_s)
            vp.Acceleration = dec(acc_mm_s2)
            dev.SetVelocityParams(vp)
            return
        except Exception:
            continue
    raise RuntimeError("VelocityParameters not found. Please verify your Kinesis build.")

def move_relative_du(dev, delta_mm: float, timeout_s=3600):
    """Attempt relative move using the most compatible command sequence (DU=int)."""
    du = int(round(delta_mm * DU_PER_MM))
    # 1) SetMoveRelativeDistance(int) → MoveRelative()
    try:
        dev.SetMoveRelativeDistance(du)
        dev.MoveRelative()
        wait_complete(dev, timeout_s)
        return "SetMoveRelativeDistance→MoveRelative"
    except Exception as e1:
        last1 = e1
    # 2) MoveRelative(int)
    try:
        dev.MoveRelative(du)
        wait_complete(dev, timeout_s)
        return "MoveRelative(int)"
    except Exception as e2:
        last2 = e2
    # 3) Use absolute move as fallback
    cur_du = int(round(get_pos_mm(dev) * DU_PER_MM))
    tgt_du = cur_du + du
    try:
        dev.SetMoveAbsolutePosition(tgt_du)
        try: dev.MoveAbsolute()
        except Exception: dev.MoveAbsolute(tgt_du)
        wait_complete(dev, timeout_s)
        return "SetMoveAbsolutePosition→MoveAbsolute"
    except Exception as e3:
        pass
    # 4) MoveTo(int)
    try:
        dev.MoveTo(tgt_du)
        wait_complete(dev, timeout_s)
        return "MoveTo(int)"
    except Exception as e4:
        raise RuntimeError(
            "All movement APIs failed.\n"
            f"1) SetMoveRelativeDistance→MoveRelative: {last1}\n"
            f"2) MoveRelative(int): {last2}\n"
            f"3) SetMoveAbsolutePosition→MoveAbsolute: {e3}\n"
            f"4) MoveTo(int): {e4}"
        )

# ---------- Execution ----------
print("[INFO] Please arm ThorCam in 'Standard / On High' mode and keep it ready.")
sample = connect_kst_robust(SER_SAMPLE)
camera = connect_kst_robust(SER_CAMERA)

# Velocity setup (slightly larger acceleration for easier start)
set_speed_mm_s(sample, sample_speed_mm, max(1.0, sample_speed_mm*5))
set_speed_mm_s(camera, camera_speed_mm, max(1.0, camera_speed_mm*5))

s0 = get_pos_mm(sample); c0 = get_pos_mm(camera)
print(f"[PLAN] sample={sample_dist_mm:.3f} mm @ {sample_speed_mm:.5f} mm/s (T≈{move_time_s:.1f}s)")
print(f"[PLAN] camera≈{camera_dist_mm:.3f} mm @ {camera_speed_mm:.5f} mm/s")
print(f"[POS ] start: sample={s0:.6f} mm, camera={c0:.6f} mm")

# Wait 1s for ThorCam ARM stabilization (important)
time.sleep(1.0)

# Start both simultaneously
ret = {"s": None, "c": None, "e1": None, "e2": None}
def run1():
    try:
        ret["s"] = move_relative_du(sample, sample_dist_mm, timeout_s=move_time_s*2+60)
    except Exception as e: ret["e1"] = e

def run2():
    try:
        ret["c"] = move_relative_du(camera, camera_dist_mm, timeout_s=move_time_s*2+60)
    except Exception as e: ret["e2"] = e

t1 = threading.Thread(target=run1); t2 = threading.Thread(target=run2)
t1.start(); t2.start(); t1.join(); t2.join()

s1 = get_pos_mm(sample); c1 = get_pos_mm(camera)
print(f"[POS ] end  : sample={s1:.6f} mm, camera={c1:.6f} mm")
print(f"[MOVE] delta: sample={s1-s0:.3f} mm, camera={c1-c0:.3f} mm")
print(f"[API ] used : sample={ret['s']}, camera={ret['c']}")
if ret["e1"] or ret["e2"]:
    print("[ERR ]", ret["e1"], ret["e2"])

# Cleanup
for d in (sample, camera):
    try: d.StopPolling()
    except Exception: pass
    try: d.Disconnect()
    except Exception: pass
print("[DONE] Minimal simultaneous relative movement test completed.")


### Return to imaging start position

The stages will return to their original positions by the total travel distance.

In [6]:
# ===== Fast "Return to Start Position": with Direction Correction Flags and Guaranteed Disconnect =====
import time, threading
import clr
from pathlib import Path
from System import Decimal as SysDecimal
from System.Globalization import CultureInfo

def mkdec(x) -> SysDecimal:
    return SysDecimal.Parse(str(x), CultureInfo.InvariantCulture)

# --- User environment ---
KINESIS_DIR = Path(r"C:\Program Files\Thorlabs\Kinesis")
SER_SAMPLE  = "26006224"
SER_CAMERA  = "26006304"
DU_PER_MM_F = 2008624.0  # float is fine (used for relative DU conversion)

# Direction correction for each axis (“+DU” may not correspond to forward direction)
# If the motion is reversed, set to -1.
DIR_SAMPLE = +1
DIR_CAMERA = +1

# --- DLLs and types ---
clr.AddReference(str(KINESIS_DIR / "Thorlabs.MotionControl.DeviceManagerCLI.dll"))
clr.AddReference(str(KINESIS_DIR / "Thorlabs.MotionControl.GenericMotorCLI.dll"))
clr.AddReference(str(KINESIS_DIR / "Thorlabs.MotionControl.KCube.StepperMotorCLI.dll"))
from Thorlabs.MotionControl.DeviceManagerCLI import DeviceManagerCLI
from Thorlabs.MotionControl.KCube.StepperMotorCLI import KCubeStepper
from Thorlabs.MotionControl.GenericMotorCLI.ControlParameters import VelocityParameters

def reconnect_kcube(serial: str, tries=6, pause=0.5):
    last=None
    for i in range(tries):
        try:
            DeviceManagerCLI.BuildDeviceList()
            d = KCubeStepper.CreateKCubeStepper(serial)
            time.sleep(0.2)
            d.Connect(serial)
            try: d.WaitForSettingsInitialized(5000)
            except: pass
            time.sleep(0.2); d.StartPolling(200)
            time.sleep(0.2); d.EnableDevice()
            time.sleep(0.3)
            try: d.LoadMotorConfiguration(serial)
            except: pass
            print(f"[OK] Connected {serial}")
            return d
        except Exception as e:
            last=e; time.sleep(pause)
    raise last

def set_speed_mm(dev, vel_mm_s: float, acc_mm_s2: float):
    vp = VelocityParameters()
    try:
        vp.MaxVelocity  = mkdec(vel_mm_s)
        vp.Acceleration = mkdec(acc_mm_s2)
        dev.SetVelocityParams(vp)
        return
    except Exception:
        pass
    # If real units are not supported, fallback to DU conversion
    DU = mkdec(DU_PER_MM_F)
    vp.MaxVelocity  = SysDecimal.Multiply(mkdec(vel_mm_s),  DU)
    vp.Acceleration = SysDecimal.Multiply(mkdec(acc_mm_s2), DU)
    dev.SetVelocityParams(vp)

def get_pos_mm(dev) -> float:
    return float(str(dev.Position))

def wait_complete(dev, timeout_s=1800):
    try:
        dev.WaitForMoveToComplete(int(timeout_s*1000))
        return
    except Exception:
        pass
    t0=time.time()
    while time.time()-t0 < timeout_s:
        try:
            if hasattr(dev,"IsDeviceBusy") and not bool(dev.IsDeviceBusy):
                break
        except Exception:
            pass
        time.sleep(0.02)

def move_relative_mm_robust(dev, delta_mm: float, dir_factor: int = +1, timeout_s=1800):
    """
    Robust relative move procedure with direction correction.
    1) Relative DU (int) using dir_factor → 2) Absolute mm (fallback) → 3) Absolute DU (fallback)
    """
    # ---- 1) Relative DU ----
    # [Modification] Removed previous '* -1' inversion.
    du = int(round(delta_mm * DU_PER_MM_F * dir_factor))

    try:
        dev.SetMoveRelativeDistance(du)
        dev.MoveRelative()
        wait_complete(dev, timeout_s)
        return "SetMoveRelativeDistance(int)→MoveRelative()"
    except Exception as e1:
        last1=e1
    try:
        dev.MoveRelative(du)
        wait_complete(dev, timeout_s)
        return "MoveRelative(int)"
    except Exception as e2:
        last2=e2

    # ---- 2) Absolute mm (Decimal) fallback ----
    try:
        # This fallback replicates the intended relative move using absolute position
        tgt_mm = get_pos_mm(dev) + (delta_mm * dir_factor)
        dev.MoveTo(mkdec(tgt_mm))
        wait_complete(dev, timeout_s)
        return "MoveTo(Decimal)"
    except Exception as e3:
        last3=e3

    # ---- 3) Absolute DU (int) fallback ----
    try:
        cur_du = int(round(get_pos_mm(dev) * DU_PER_MM_F))
        tgt_du = cur_du + du
        dev.MoveTo(tgt_du)
        wait_complete(dev, timeout_s)
        return "MoveTo(int)"
    except Exception as e4:
        raise RuntimeError(
            "No usable API found for return motion.\n"
            f"1) SetMoveRelativeDistance→MoveRelative: {last1}\n"
            f"2) MoveRelative(int): {last2}\n"
            f"3) MoveTo(Decimal): {last3}\n"
            f"4) MoveTo(int): {e4}"
        )

# ===== Execution =====
sample = camera = None
try:
    sample = reconnect_kcube(SER_SAMPLE)
    camera = reconnect_kcube(SER_CAMERA)

    s0 = get_pos_mm(sample); c0 = get_pos_mm(camera)
    print(f"[POS start] sample={s0:.6f} mm, camera={c0:.6f} mm")

    # Fast return velocity (adjust if necessary)
    set_speed_mm(sample, vel_mm_s=1.0, acc_mm_s2=5.0)
    set_speed_mm(camera, vel_mm_s=0.8, acc_mm_s2=4.0)

    # Return distance (equal to the previous travel distance)
    delta_sample = -13.0
    delta_camera = -(13.0 * 0.342)
    print(f"[MOVE back] sample {delta_sample:+.3f} mm, camera {delta_camera:+.3f} mm")

    res = {}
    t1 = threading.Thread(target=lambda: res.update(sample=move_relative_mm_robust(sample, delta_sample, DIR_SAMPLE)))
    t2 = threading.Thread(target=lambda: res.update(camera=move_relative_mm_robust(camera, delta_camera, DIR_CAMERA)))
    t1.start(); t2.start(); t1.join(); t2.join()

    s1 = get_pos_mm(sample); c1 = get_pos_mm(camera)
    print(f"[API] sample={res.get('sample')}, camera={res.get('camera')}")
    print(f"[POS end] sample={s1:.6f} mm, camera={c1:.6f} mm")
    print(f"[DELTA] moved: sample={s1 - s0:.3f} mm, camera={c1 - c0:.3f} mm")

finally:
    for dev, name in [(sample,"sample"), (camera,"camera")]:
        if dev is None: continue
        try: dev.StopPolling()
        except Exception: pass
        try:
            dev.Disconnect()
            print(f"[CLOSE] {name} disconnected.")
        except Exception: pass

print("[DONE] Return motion complete. All devices safely disconnected.")


[OK] Connected 26006224
[OK] Connected 26006304
[POS start] sample=13.949892 mm, camera=21.023208 mm
[MOVE back] sample -13.000 mm, camera -4.446 mm
[API] sample=MoveRelative(int), camera=MoveRelative(int)
[POS end] sample=0.949892 mm, camera=16.577208 mm
[DELTA] moved: sample=-13.000 mm, camera=-4.446 mm
[CLOSE] sample disconnected.
[CLOSE] camera disconnected.
[DONE] 戻し完了。全デバイス切断済み。
